# Sentiment classification on DailyDialog (Multinomial Naive Bayes)

This notebook reproduces `sentiment-convo-dailydialog-MNB.py` (week3) but reads the raw DailyDialog data files directly instead of using `convokit`.

It expects the following file to already be available locally in the Colab runtime, under `/content/sample_data/` (the standard Colab `sample_data` folder, visible in the Files pane):
- `dailydialog_lines.txt` (columns: `line_id`, `speaker_id`, `text`, `sentiment`, where `sentiment` is `0` for negative and `1` for positive)

Drag this file directly into the `sample_data` folder in the Colab Files pane before running the cells below.

We then:
1. Split utterances into positive vs. negative texts based on the `sentiment` label.
2. Vectorize the texts with TF-IDF.
3. Train a Multinomial Naive Bayes classifier to predict sentiment from the text.
4. Inspect the most salient words per class and evaluate precision, recall and accuracy.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.naive_bayes import MultinomialNB

## 1. Load the data

We read the lines file and split the utterance texts into `pos` and `neg` lists based on the `sentiment` column. This replaces the `convokit.Corpus` used in the original script.

In [ ]:
LINES_PATH = "/content/sample_data/dailydialog_lines.txt"


def process_data():
    print("Loading data...")
    pos = []
    neg = []
    with open(LINES_PATH, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 4:
                continue
            _line_id, _speaker_id, text, sentiment = fields[0], fields[1], fields[2], fields[3]
            if sentiment == "0":
                neg.append(text)
            elif sentiment == "1":
                pos.append(text)
    print("Done.")
    print("No. pos", len(pos), "No. neg:", len(neg))
    return pos[:1000], neg[:1000]


pos, neg = process_data()

## 2. Vectorize and prepare train/test splits

Negative utterances are labeled `0.0`, positive utterances `1.0`. We fit a TF-IDF vectorizer on all utterances, then vectorize the train/test splits separately, as in the original script.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
vectorizer.fit(pos + neg)

labeled_utt = [(n, 0.0) for n in neg] + [(p, 1.0) for p in pos]
labels = [l for (_utt, l) in labeled_utt]

labeled_utt_train, labeled_utt_test, labels_train, labels_test = train_test_split(
    labeled_utt, labels, test_size=0.1
)

X_train = []
X_test = []
y_train = []
y_test = []
text_test = []

for utt, label in labeled_utt_train:
    values = list(vectorizer.transform([utt]).toarray()[0])
    X_train.append(values)
    y_train.append(label)

for utt, label in labeled_utt_test:
    values = list(vectorizer.transform([utt]).toarray()[0])
    X_test.append(values)
    y_test.append(label)
    text_test.append(utt)

## 3. Train the Multinomial Naive Bayes classifier

In [ ]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

## 4. Most salient words per class

Rank the vocabulary by log-probability for each class (0 = negative, 1 = positive) and save the top-100 words per class.

In [ ]:
def get_salient_words(model, vectorizer, class_id):
    words = vectorizer.get_feature_names_out()
    zipped = list(zip(words, model.feature_log_prob_[class_id]))
    sorted_zip = sorted(zipped, key=lambda t: t[1], reverse=True)
    return sorted_zip


with open("dailydialog-salient-words-100.txt", "w") as fp_w:
    fp_w.write("Neg:" + str(get_salient_words(clf, vectorizer, 0)[:100]) + "\n\n")
    fp_w.write("Pos:" + str(get_salient_words(clf, vectorizer, 1)[:100]))
print("See dailydialog-salient-words-100.txt for top-100 most salient words per class.")

## 5. Evaluate on the test set

In [ ]:
y_pred = clf.predict(X_test)
n = 0
for pred in y_pred:
    print(text_test[n], y_test[n], pred)
    n += 1

print("Precision:", metrics.precision_score(y_test, y_pred))
print("Recall:", metrics.recall_score(y_test, y_pred))
print("Accuracy:", metrics.accuracy_score(y_test, y_pred))